# RQ2 — Audit offline delle traiettorie storiche V3

Eseguire **dopo** i 12 test PyCharm. Il notebook legge solo JSON nella directory congelata dei risultati V3 su Google Drive (non apre `Market.zip`, `record.csv`, scorer o casi OOD). Calcola aggregati sulla compatibilità con il nuovo *evidence bundle*, senza stampare nomi di casi, risposte, KPI o predizioni. Non modifica nulla.

Questo serve a rispondere alla domanda di **fattibilità software**: quali proporzioni di traiettorie già catturate possono essere ricostruite senza inventare l'associazione fra chiamata e risultato dei tool? I risultati storici non diventeranno esempi di discovery RQ2 né materiale per generare pseudoregole.


In [ ]:
# RQ2 — Audit GT-free su traiettorie storiche V3, ESCLUSIVAMENTE in lettura.
# Nessun caso OOD, nessuna metrica di accuratezza o score, nessuna API.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
ROOT = Path("/content/drive/MyDrive/Tesi_OpenRCA")
RESULTS = ROOT / "data/results/v2/openrca_harness_v2_five_agents_development_v3"
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
if not RESULTS.is_dir():
    raise FileNotFoundError(
        "Frozen V3 results directory not found. No GT or OOD source substituted: "
        + str(RESULTS)
    )
if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Invalid clone path; no deletion or overwrite")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        text=True,
    ).strip() != BRANCH:
        raise RuntimeError("Wrong branch; no automatic checkout")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Local changes in clone; no automatic pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        capture_output=True, check=True, text=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", BRANCH, URL, str(REPO)],
        capture_output=True, check=True, text=True,
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
result = subprocess.run(
    [sys.executable, "-m", "evaluation.rq2_historical_trajectory_audit",
     "--data-dir", str(ROOT)],
    cwd=REPO, env=env, text=True, capture_output=True,
)
if result.returncode:
    raise RuntimeError("GT-free historical audit failed:\n" + result.stderr[-5000:])
report = json.loads(result.stdout)
if not (
    report.get("audit") == "rq2_historical_trajectory_feasibility_v0"
    and report.get("status") == "HISTORICAL_COMPATIBILITY_ONLY"
    and report.get("read_only") is True
    and report.get("read_ground_truth") is False
    and report.get("read_scoring") is False
    and report.get("read_ood_cases") is False
    and report.get("read_model_predictions_for_tuning") is False
    and report.get("model_api_calls") == 0
    and report.get("files_written") is False
):
    raise RuntimeError("Read-only / GT-free invariants violated")
print("=== RQ2 — COMPATIBILITÀ DELLE TRAIETTORIE STORICHE V3 ===")
print("Branch:", BRANCH)
print("Commit:", commit)
print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nNO OOD CASES, NO SCORING, NO API, NO FILE WRITES.")


## Dopo l'esecuzione

Incollare il riepilogo completo. L'assenza della directory o dei file result è un problema di percorso/layout da verificare; non sostituire la directory con `data/results/scoring` o dati OOD. Nessuna autorizzazione API è implicata.
